# Week 9 — Wednesday Lab Notebook  
## Topic: Model Selection

This lab is for **Week 9 Wednesday**.  
Today we will learn how to choose a better model in a more correct way.

We will cover:

- cross-validation
- choosing metrics
- GridSearchCV
- RandomizedSearchCV
- learning curves idea
- validation strategy

## 3-Hour Structure

**Hour 1**
- Understand model selection in simple words
- Why one train/test split is sometimes not enough
- Cross-validation and folds
- Validation strategy basics

**Hour 2**
- Choosing the right metric
- Compare models using cross-validation
- Use `GridSearchCV`
- Use `RandomizedSearchCV`

**Hour 3**
- Understand learning curves
- Read training score vs validation score
- Discuss common mistakes
- Mini in-class practice
- After-lab tasks

## Part A — What is Model Selection?

Model selection means:

**From many possible models or many possible settings, choose the one that is best for the problem.**

In simple words, suppose you have:

- Logistic Regression
- kNN
- Decision Tree
- Random Forest

Now the question is:

**Which one should we use?**

Also, even inside one model, there can be many settings.

Example:

For kNN:
- `n_neighbors = 3`
- `n_neighbors = 5`
- `n_neighbors = 7`

For Decision Tree:
- `max_depth = 2`
- `max_depth = 4`
- `max_depth = 8`

So model selection is not only:
- choosing the model

It is also:
- choosing the best settings for that model

## Part B — Why Not Just Train Once and Stop?

Many beginners do this:

1. split data into train and test
2. train one model
3. check accuracy
4. stop

This is useful, but sometimes it is **not enough**.

Why?

Because maybe that one split was:
- lucky
- unlucky
- not fully representative

A model may look very good on one split, but weaker on another split.

That is why we use more reliable methods like:

- cross-validation
- proper validation strategy
- metric comparison
- hyperparameter search

## Part C — Validation Strategy in Simple Words

A validation strategy means:

**What exact method will we use to judge whether a model is good or not?**

A good validation strategy helps us answer:

- how should we split the data?
- which metric should we use?
- how do we tune parameters?
- how do we avoid unfair comparison?
- how do we avoid leakage?

A simple and common plan is:

1. keep a test set separate
2. use the training data for cross-validation
3. tune models on training folds only
4. use the final test set only at the end

This is much safer than checking the test set again and again.

## Part D — Cross-Validation in Simple Words

Cross-validation means:

**We divide the training data into multiple parts, train multiple times, and average the result.**

The most common type is **k-fold cross-validation**.

Example: 5-fold cross-validation

- split data into 5 parts
- use 4 parts for training
- use 1 part for validation
- repeat this 5 times
- each part becomes validation once

Then we calculate the average score.

This gives a more stable idea of model performance.

## Part E — Why Cross-Validation is Helpful

Cross-validation helps because:

- it uses data more carefully
- it reduces dependence on one lucky split
- it gives average performance
- it helps compare models more fairly
- it is useful for hyperparameter tuning

Important idea:

A model with one high score is not always the best.  
A model with **good average score** and **stable performance** is often safer.

## Part F — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV,
    learning_curve
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

np.random.seed(42)

## Part G — Load a Real Dataset

We will use the **Breast Cancer dataset** from scikit-learn.

This is a classification dataset.

Target values:
- `0` = malignant
- `1` = benign

We will use it because:
- it is already available
- it is not too large
- it is good for model comparison

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X.head()

## Part H — Inspect the Dataset

In [ ]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print()
print("Target counts:")
print(y.value_counts())

In [ ]:
X.info()

### What should students observe?

- `X` has many numeric columns
- `y` is the target column
- this is a **binary classification** problem
- the classes are not exactly equal, so metric choice matters

## Part I — Train/Test Split

We will keep aside a test set.

Important rule:

**Do not use the test set again and again while tuning.**

We keep it for final checking.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

## Part J — Build a Few Candidate Models

Today we will compare these models:

- Logistic Regression
- kNN
- Decision Tree
- Random Forest

For fair comparison:
- models that need scaling should get scaling
- tree-based models usually do not need scaling

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, random_state=42))
    ]),
    "kNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

## Part K — Check One Simple Train/Test Score First

Before cross-validation, let us see a simple test score.

In [ ]:
simple_results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    simple_results.append([name, acc])

simple_results_df = pd.DataFrame(simple_results, columns=["Model", "Test Accuracy"])
simple_results_df.sort_values("Test Accuracy", ascending=False)

### Teaching point

This table is useful, but it is still based on **one split only**.

Now we will use cross-validation to get a more reliable view.

## Part L — 5-Fold Cross-Validation

We will use `cross_val_score()` on the training data only.

Why training data only?

Because the test set should stay untouched until the end.

In [ ]:
cv_results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )
    cv_results.append([
        name,
        scores.mean(),
        scores.std(),
        scores
    ])

cv_df = pd.DataFrame(cv_results, columns=["Model", "Mean CV Accuracy", "Std CV Accuracy", "Fold Scores"])
cv_df.sort_values("Mean CV Accuracy", ascending=False)

## Part M — Read the Cross-Validation Table

Focus on:

- **Mean CV Accuracy** → average performance
- **Std CV Accuracy** → stability

Interpretation idea:

- high mean + low std = stronger and more stable
- high mean + high std = good, but unstable
- low mean = weaker model

So when choosing a model, do not see only one number.  
Look at both performance and consistency.

## Part N — Understand Folds with StratifiedKFold

For classification, we often use **StratifiedKFold**.

It tries to keep class balance similar in each fold.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_no = 1
for train_index, valid_index in skf.split(X_train, y_train):
    y_train_fold = y_train.iloc[train_index]
    y_valid_fold = y_train.iloc[valid_index]

    print(f"Fold {fold_no}")
    print("  Train class counts:", dict(y_train_fold.value_counts().sort_index()))
    print("  Valid class counts:", dict(y_valid_fold.value_counts().sort_index()))
    fold_no += 1

### Simple meaning

StratifiedKFold is helpful when:
- classes are not perfectly equal
- we want fair folds
- we want more reliable validation

## Part O — Choosing the Right Metric

Different problems need different metrics.

### For classification:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

### For regression:
- MAE
- MSE
- RMSE
- R²

Important rule:

**Do not choose a metric blindly.**

Example:
- If false negatives are very dangerous, recall may matter more.
- If both precision and recall matter, F1-score may be useful.
- If classes are balanced and mistake cost is simple, accuracy may be okay.

## Part P — Compare Metrics on the Test Set

We will compare two models:
- Logistic Regression
- Random Forest

In [ ]:
log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

rf_model = RandomForestClassifier(random_state=42)

log_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

log_preds = log_model.predict(X_test)
rf_preds = rf_model.predict(X_test)

log_probs = log_model.predict_proba(X_test)[:, 1]
rf_probs = rf_model.predict_proba(X_test)[:, 1]

metric_table = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "Logistic Regression": [
        accuracy_score(y_test, log_preds),
        precision_score(y_test, log_preds),
        recall_score(y_test, log_preds),
        f1_score(y_test, log_preds),
        roc_auc_score(y_test, log_probs)
    ],
    "Random Forest": [
        accuracy_score(y_test, rf_preds),
        precision_score(y_test, rf_preds),
        recall_score(y_test, rf_preds),
        f1_score(y_test, rf_preds),
        roc_auc_score(y_test, rf_probs)
    ]
})

metric_table

## Part Q — Confusion Matrix Reminder

In [ ]:
print("Confusion Matrix - Logistic Regression")
print(confusion_matrix(y_test, log_preds))
print()
print("Confusion Matrix - Random Forest")
print(confusion_matrix(y_test, rf_preds))

### Teaching point

Sometimes two models have similar accuracy, but:

- one has better recall
- one has better precision
- one has better ROC-AUC

So the “best” model depends on:
- the real-world goal
- the cost of mistakes
- the chosen metric

## Part R — Hyperparameters in Simple Words

Hyperparameters are settings we choose before training.

Examples:

For kNN:
- `n_neighbors`

For Decision Tree:
- `max_depth`
- `min_samples_split`

For Random Forest:
- `n_estimators`
- `max_depth`
- `min_samples_leaf`

We do not guess these blindly.

We use search methods like:
- GridSearchCV
- RandomizedSearchCV

## Part S — GridSearchCV in Simple Words

Grid search means:

**Try all combinations from a small parameter grid.**

Example:
- `n_neighbors = [3, 5, 7]`
- `weights = ['uniform', 'distance']`

Then the tool tries every combination and finds the best one.

## Part T — Use GridSearchCV with kNN

In [ ]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)
print()
print("Best CV score:")
print(grid_search.best_score_)

## Part U — View Grid Search Results

In [ ]:
grid_results_df = pd.DataFrame(grid_search.cv_results_)

grid_results_df = grid_results_df[[
    "params",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]].sort_values("rank_test_score")

grid_results_df.head(10)

### What should students notice?

- Every row is one parameter combination
- `mean_test_score` is the average CV score
- `rank_test_score = 1` means best setting
- Grid search is thorough, but can become slow if the grid is too large

## Part V — RandomizedSearchCV in Simple Words

Randomized search means:

**Try only some random combinations instead of all combinations.**

This is useful when:
- parameter space is large
- full grid search is expensive
- we want faster search

## Part W — Use RandomizedSearchCV with Random Forest

In [ ]:
param_dist = {
    "n_estimators": [50, 100, 150, 200, 300],
    "max_depth": [None, 3, 5, 7, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=12,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best parameters:")
print(random_search.best_params_)
print()
print("Best CV score:")
print(random_search.best_score_)

## Part X — View Random Search Results

In [ ]:
random_results_df = pd.DataFrame(random_search.cv_results_)

random_results_df = random_results_df[[
    "params",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]].sort_values("rank_test_score")

random_results_df.head(10)

## Part Y — Compare Best Models

Now let us compare:
- best kNN from GridSearchCV
- best Random Forest from RandomizedSearchCV
- simple Logistic Regression baseline

In [ ]:
best_knn = grid_search.best_estimator_
best_rf = random_search.best_estimator_

comparison_models = {
    "Logistic Regression": log_model,
    "Best kNN": best_knn,
    "Best Random Forest": best_rf
}

comparison_rows = []

for name, model in comparison_models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    comparison_rows.append([
        name,
        accuracy_score(y_test, preds),
        precision_score(y_test, preds),
        recall_score(y_test, preds),
        f1_score(y_test, preds)
    ])

comparison_df = pd.DataFrame(
    comparison_rows,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1"]
)

comparison_df.sort_values("F1", ascending=False)

### Teaching point

This is closer to real model selection.

We:
- made candidate models
- used cross-validation
- tuned parameters
- compared metrics
- then checked final performance

## Part Z — Learning Curves Idea

A learning curve shows how model performance changes when training size increases.

Usually we look at:
- training score
- validation score

This helps us understand:
- underfitting
- overfitting
- whether more data may help

## Part AA — Draw a Learning Curve

We will use a Decision Tree because it often shows a clear gap between training score and validation score.

In [ ]:
train_sizes, train_scores, valid_scores = learning_curve(
    DecisionTreeClassifier(random_state=42),
    X_train,
    y_train,
    cv=5,
    scoring="accuracy",
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
valid_mean = valid_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, marker="o", label="Training Score")
plt.plot(train_sizes, valid_mean, marker="o", label="Validation Score")
plt.xlabel("Training Set Size")
plt.ylabel("Accuracy")
plt.title("Learning Curve - Decision Tree")
plt.legend()
plt.grid(True)
plt.show()

## Part AB — How to Read the Learning Curve

### Case 1: Both scores low
This can mean **underfitting**.

### Case 2: Training high, validation much lower
This can mean **overfitting**.

### Case 3: Both scores good and close
This is usually a healthier sign.

### Case 4: Validation score still rising
Maybe more data can still help.

## Part AC — A Simple Validation Strategy Checklist

When doing a proper ML experiment, try to follow this order:

1. understand the problem type
2. choose the metric carefully
3. split the test set once
4. use training data for cross-validation
5. tune hyperparameters on training folds only
6. compare candidate models fairly
7. use the test set only at the end
8. report results clearly

This is a more professional workflow.

## Part AD — Common Validation Strategies

### 1. Train/Validation/Test Split
Good for many beginner cases.

### 2. k-Fold Cross-Validation
Good when data is limited and we want stronger comparison.

### 3. Stratified k-Fold
Very useful for classification.

### 4. Time-based split
Used in time-series problems.  
We should not randomly shuffle future and past data there.

### 5. Group-based split
Useful when samples from the same person, machine, or group should stay together.

## Part AE — Important Warning About Leakage

Data leakage happens when information from outside the training fold enters the model unfairly.

Examples:
- scaling before proper split
- encoding before proper split
- selecting features using full data before validation
- checking the test set many times during tuning

Leakage gives unrealistically good results.

So always be careful.

## Part AF — Common Beginner Mistakes

1. choosing model from test set again and again  
2. tuning on the test set  
3. using only accuracy for every problem  
4. ignoring class imbalance  
5. forgetting scaling for kNN or Logistic Regression  
6. using too large a parameter grid without reason  
7. reporting only best score, not method  
8. not fixing `random_state` when comparison needs reproducibility  
9. comparing models with different validation methods  
10. not understanding what metric actually means

## Part AG — Mini In-Class Practice

Do these in class:

1. Run 5-fold cross-validation for a `DecisionTreeClassifier`.
2. Change the scoring metric from `"accuracy"` to `"f1"`.
3. Run a small grid search for Decision Tree using `max_depth = [2, 4, 6, 8]`.
4. Compare Logistic Regression and kNN using mean CV accuracy.
5. Draw a learning curve for Logistic Regression and compare it with Decision Tree.

## Part AH — Recap

Today we learned:

- what model selection means
- why one split is not always enough
- cross-validation
- metric choice
- GridSearchCV
- RandomizedSearchCV
- learning curve idea
- validation strategy
- leakage warnings

This is a very important topic because it helps us make **more correct ML decisions**.

# After-Lab Tasks (10)

### Task 1
Load the Breast Cancer dataset and display:
- shape of features
- shape of target
- first 5 rows

### Task 2
Split the dataset into training and testing sets using:
- `test_size=0.25`
- `random_state=42`
- `stratify=y`

### Task 3
Train a `LogisticRegression` model and calculate test accuracy.

### Task 4
Train a `KNeighborsClassifier` model and calculate test accuracy.

### Task 5
Use 5-fold cross-validation on Logistic Regression and print all fold scores.

### Task 6
Create a table showing mean CV accuracy for:
- Logistic Regression
- kNN
- Decision Tree

### Task 7
Apply `GridSearchCV` on kNN for:
- `n_neighbors = [3, 5, 7, 9]`
- `weights = ['uniform', 'distance']`

Then print:
- best parameters
- best CV score

### Task 8
Apply `RandomizedSearchCV` on Random Forest with at least 3 parameters.

Then print:
- best parameters
- best CV score

### Task 9
For the final chosen model, calculate:
- accuracy
- precision
- recall
- F1-score

### Task 10
Draw a learning curve for a Decision Tree model and write 3 lines about what the graph shows.

# Optional Homework Challenge

Use the same dataset and create a **complete model selection report**.

Your report should include:

1. names of at least 4 models
2. one simple train/test comparison table
3. one cross-validation comparison table
4. one tuned model using GridSearchCV or RandomizedSearchCV
5. final metric table
6. one learning curve
7. 5 to 7 lines of conclusion:
   - which model you selected
   - why you selected it
   - which metric you trusted most
   - what you would improve next